In [ ]:
import json
import os
import sqlparse
from tqdm import tqdm
import sys






In [ ]:
sys.path.append("/kaggle/input/datasets/kotikalayashwant/t5-nl2sql")

from test_model import Text2SQLEngine
# --- Configuration ---
# Adjust these paths to point to your local Spider dataset files
DATASET_PATH = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset"
DEV_JSON_PATH = f"{DATASET_PATH}/spider/dev.json"
TABLES_JSON_PATH = f"{DATASET_PATH}/spider/tables.json"
OUTPUT_PREDICTIONS = "/kaggle/working/predictions.txt"

In [ ]:
def normalize_sql(sql):
    """
    Normalizes SQL for a baseline string comparison.
    Removes extra whitespace, standardizes case, and strips trailing semicolons.
    """
    if not sql:
        return ""
    # Use sqlparse (from your requirements) to format the query
    sql = sqlparse.format(sql, keyword_case='upper', strip_comments=True)
    return " ".join(sql.split()).strip(";").lower()

In [ ]:

def evaluate():
    if not os.path.exists(DEV_JSON_PATH) or not os.path.exists(TABLES_JSON_PATH):
        print(f"Error: Could not find Spider dataset files.")
        print(f"Please ensure {DEV_JSON_PATH} and {TABLES_JSON_PATH} exist.")
        return

    print(f"Loading Spider development dataset from {DEV_JSON_PATH}...")
    with open(DEV_JSON_PATH, "r", encoding="utf-8") as f:
        dev_data = json.load(f)

    print(f"Initializing Text2SQLEngine with schemas from {TABLES_JSON_PATH}...")
    # This will boot up your T5 model and BGE retriever
    engine = Text2SQLEngine(TABLES_JSON_PATH)

    correct_count = 0
    total_count = len(dev_data)
    predictions = []

    print(f"Starting inference on {total_count} queries...")

    # tqdm provides a progress bar so you aren't guessing if the script froze
    for item in tqdm(dev_data, desc="Evaluating"):
        question = item["question"]
        db_id = item["db_id"]
        ground_truth_sql = item["query"]

        # 1. Generate prediction using your engine
        result = engine.generate(question, db_id)

        # Handle cases where the engine returns vague or validation failures
        predicted_sql = result.get("sql")
        if predicted_sql is None:
            predicted_sql = "SELECT 1" # Fallback to prevent official script from crashing

        # 2. Save exact string format required by official Spider evaluation
        predictions.append(predicted_sql.replace('\n', ' ') + "\n")

        # 3. Calculate baseline normalized string match
        norm_predicted = normalize_sql(predicted_sql)
        norm_truth = normalize_sql(ground_truth_sql)

        if norm_predicted == norm_truth:
            correct_count += 1

    # 4. Compute and report baseline accuracy
    accuracy = (correct_count / total_count) * 100

    print("\n" + "="*40)
    print("      EVALUATION COMPLETE")
    print("="*40)
    print(f"Total Queries Evaluated : {total_count}")
    print(f"Strict String Matches   : {correct_count}")
    print(f"Baseline Accuracy       : {accuracy:.2f}%")
    print("="*40)
    print("* Note: This is a strict string match. True accuracy (EM) will be higher.")

    # 5. Export predictions
    print(f"\nSaving raw predictions to '{OUTPUT_PREDICTIONS}'...")
    with open(OUTPUT_PREDICTIONS, "w", encoding="utf-8") as f:
        f.writelines(predictions)

    print("Done! To get your final official score, run the Yale Lily Spider evaluation script:")
    print(f"python evaluation.py --gold {DEV_JSON_PATH} --pred {OUTPUT_PREDICTIONS} --etype match --table {TABLES_JSON_PATH}")



In [ ]:
evaluate()

Loading Spider development dataset from /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json...
Initializing Text2SQLEngine with schemas from /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json...
Device: cuda
Loading schemas ...
Loading BGE model: BAAI/bge-base-en-v1.5 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

Starting inference on 1034 queries...


Evaluating: 100%|██████████| 1034/1034 [17:58<00:00,  1.04s/it]


      EVALUATION COMPLETE
Total Queries Evaluated : 1034
Strict String Matches   : 85
Baseline Accuracy       : 8.22%
* Note: This is a strict string match. True accuracy (EM) will be higher.

Saving raw predictions to '/kaggle/working/predictions.txt'...
Done! To get your final official score, run the Yale Lily Spider evaluation script:
python evaluation.py --gold /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev.json --pred /kaggle/working/predictions.txt --etype match --table /kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json


In [ ]:
GOLD_FILE = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/dev_gold.sql"
PREDICTED_FILE = "/kaggle/input/datasets/kotikalayashwant/t5-nl2sql/predictions.txt"
EVALUATION_TYPE = "all"
DATABASE_DIR = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database"
TABLES_DIR = "/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json"



In [ ]:
sys.path.append("/kaggle/input/datasets/kotikalayashwant/t5-nl2sql")

In [ ]:
!cd /kaggle/input/datasets/kotikalayashwant/t5-nl2sql && python evaluation.py --gold $GOLD_FILE --pred $PREDICTED_FILE --etype $EVALUATION_TYPE --db $DATABASE_DIR --table $TABLES_DIR


eval_err_num:1
easy pred: SELECT count(*) FROM vocal
easy gold: SELECT count(*) FROM singer

eval_err_num:2
easy pred: SELECT count(*) FROM vocal
easy gold: SELECT count(*) FROM singer

eval_err_num:3
medium pred: SELECT name , country , age FROM vocal ORDER BY age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:4
medium pred: SELECT name , country , age FROM vocal ORDER BY age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:5
medium pred: SELECT avg(age) , min(age) , max(age) FROM vocal WHERE country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:6
medium pred: SELECT avg(age) , min(age) , max(age) FROM vocal WHERE country = 'France'
medium gold: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'

eval_err_num:7
medium pred: SELECT song_name , song_release_year FROM vocals ORDER BY age LIMIT 1
medium g